# Tutorial 4 — Knowledge Base and Learning

A **knowledge base** (KB) is a set of formulas whose satisfaction we want to maximise.
In sLTN, `KB([φ1, φ2, …])` aggregates all clause truth values into a single scalar via `SatAgg`.
The training objective is simply `loss = 1 − KB`.

**Key pattern**
```python
phi = sig.parse("∀ x_pos: A(x_pos)")   # parse once, outside the loop
for (x_pos_batch,), (x_neg_batch,) in zip(loader_pos, loader_neg):
    batch = {"x_pos": x_pos_batch, "x_neg": x_neg_batch}
    sat = interp(phi_kb, batch).value
    (1.0 - sat).backward()
```

This tutorial: binary classification — learn predicate `A` such that
$$\mathcal{K} = \{\,\forall x^+ A(x^+),\quad \forall x^- \lnot A(x^-)\,\}$$


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sltn
from sltn import Signature, Interpretation, KB
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1 — Dataset

200 points in $[0,1]^2$.  Positive class: inside disc of radius 0.3 centred on $(0.5, 0.5)$.

In [ ]:
n = 200
pts    = torch.rand(n, 2)
labels = ((pts - 0.5).pow(2).sum(1) < 0.09)   # True = positive

# Split 80/20
n_train = int(n * 0.8)
pts_tr, lbl_tr = pts[:n_train],  labels[:n_train]
pts_te, lbl_te = pts[n_train:],  labels[n_train:]

plt.figure(figsize=(4, 4))
plt.scatter(*pts_tr[lbl_tr].T,  label="pos", c="C0", alpha=0.6, s=15)
plt.scatter(*pts_tr[~lbl_tr].T, label="neg", c="C1", alpha=0.6, s=15)
plt.legend(); plt.title("Training data"); plt.tight_layout(); plt.show()

## 2 — Signature, formulas, and neural model

Declare once.  Formulas are parsed outside the training loop — they are pure syntax trees.

In [ ]:
# Declare signature
sig   = Signature("clf")
sig.sort("Point")
sig.variable("x_pos", "Point")
sig.variable("x_neg", "Point")
sig.predicate("A", ["Point"])

# Declare interpretation
interp = Interpretation(sig).to(device)
interp["Point"] = sltn.Type("Point", shape=(2,))  # 2-D real vectors

# Neural model
class ModelA(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ELU(),
            nn.Linear(16, 16), nn.ELU(),
            nn.Linear(16, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)

interp["A"] = ModelA()   # auto-moved to device

# Formulas — declared once, outside any training loop
phi_pos = sig.parse("forall x_pos: A(x_pos)")
phi_neg = sig.parse("forall x_neg: not A(x_neg)")
phi_kb  = KB([phi_pos, phi_neg])
print(phi_kb)

### `KB([...])` vs the `{ φ1, φ2, … }` brace syntax

`KB([phi_pos, phi_neg])` (used above) and the string syntax `sig.parse("{ φ1, φ2 }")` build the
same object. The brace form is convenient when the whole KB is expressed as one formula string.

**KB clauses must be closed** (no free variables/structural variables) — this is checked when a
`{ … }` KB is parsed. An open clause raises a `ValueError` immediately, rather than failing later
with a confusing shape mismatch during evaluation.

In [ ]:
# Brace syntax builds the same KB as KB([phi_pos, phi_neg])
phi_kb_brace = sig.parse("{ forall x_pos: A(x_pos), forall x_neg: not A(x_neg) }")
print("Same clauses:", [str(c) for c in phi_kb_brace.clauses] == [str(c) for c in phi_kb.clauses])

# An open clause (free variable x_pos) is rejected at parse time
try:
    sig.parse("{ A(x_pos) }")
except ValueError as e:
    print("Rejected open KB clause:", e)

## 3 — Training

Create separate DataLoaders and pass per-batch tensors explicitly via the assignment dict.


In [ ]:
BATCH = 64
loader_pos = DataLoader(TensorDataset(pts_tr[lbl_tr]),  batch_size=BATCH, shuffle=True)
loader_neg = DataLoader(TensorDataset(pts_tr[~lbl_tr]), batch_size=BATCH, shuffle=True)

opt = torch.optim.Adam(interp.parameters(), lr=1e-3)

EPOCHS = 1000
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for (x_pos_batch,), (x_neg_batch,) in zip(loader_pos, loader_neg):
        opt.zero_grad()
        batch = {"x_pos": x_pos_batch.to(device), "x_neg": x_neg_batch.to(device)}
        sat  = interp(phi_kb, batch).value
        loss = 1.0 - sat
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    if epoch % 50 == 0:
        print(f"epoch {epoch:4d}  loss={epoch_loss:.4f}")


### 3b — The `kb_describe` / `kb_backward` training helpers

`sltn.tools.optim` provides an alternative to the manual `loss.backward()` loop above.

- `kb_describe(kb)` labels the clauses (`φ_0`, `φ_1`, …) for logging and returns a dict to reuse.
- `kb_backward(optimizer, interp, kb_dict)` evaluates every clause in one forward pass and
  backpropagates `1 - SatAgg(...)`. By default (`aggregator="satagg"`, used below) this is exactly
  the same scalar-loss computation as the manual loop above — no extra dependency is required.
  Passing a different `aggregator` (e.g. `"mgda"`, `"pcgrad"`, `"upgrad"`) instead treats each
  clause as a separate objective and combines their gradients with
  [TorchJD](https://torchjd.org/stable/), a multi-objective optimization library based on Jacobian
  Descent (https://arxiv.org/html/2406.16232v1); this mode requires the optional
  `sltn[multiobjective]` extra and is not demonstrated in this tutorial.
  `kb_backward` reads whatever is currently bound in `interp` — bind the batch tensors via
  `interp[x] = ...` before calling it (there is no separate assignment-dict argument, unlike
  `interp(phi, batch)`).
- `kb_eval(interp, kb_dict)` evaluates the clauses under `torch.no_grad()` for monitoring only.

This is the same pattern used in the project README's quickstart.

In [ ]:
from sltn import kb_backward, kb_describe, kb_eval

# A separate interpretation/model, trained the same way but via kb_backward/kb_describe
interp_helper = Interpretation(sig).to(device)
interp_helper["Point"] = sltn.Type("Point", shape=(2,))
interp_helper["A"] = ModelA().to(device)

kb_dict = kb_describe(phi_kb)   # labels the clauses φ_0, φ_1, … (prints once)

opt_helper = torch.optim.Adam(interp_helper.parameters(), lr=1e-3)

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for (x_pos_batch,), (x_neg_batch,) in zip(loader_pos, loader_neg):
        opt_helper.zero_grad()
        interp_helper["x_pos"] = x_pos_batch.to(device)
        interp_helper["x_neg"] = x_neg_batch.to(device)
        stats = kb_backward(opt_helper, interp_helper, kb_dict, aggregator="satagg")
        opt_helper.step()
        epoch_loss += stats["agg_loss"]
    if epoch % 50 == 0:
        print(f"epoch {epoch:4d}  agg_loss={epoch_loss:.4f}  agg_sat={stats['agg_sat']:.4f}")

# kb_eval mirrors kb_backward but under torch.no_grad(), for monitoring only
print("\nFinal per-clause satisfaction (kb_eval):", dict(kb_eval(interp_helper, kb_dict)))

# named_parameters() pairs each learnable tensor with the symbol name it came from
print("\nnamed_parameters():")
for name, p in interp_helper.named_parameters():
    print(f"  {name:<20s} shape={tuple(p.shape)}")

## 4 — Evaluation

Evaluate the learned predicate on a dense grid to visualise the decision boundary.

In [ ]:
g = torch.linspace(0, 1, 100)
gx, gy = torch.meshgrid(g, g, indexing="ij")
grid   = torch.stack([gx.flatten(), gy.flatten()], dim=1).to(device)

# Declare a fresh variable for the grid and assign the tensor via interp[x_grid] = grid
x_grid = sig.variable("x_grid", "Point")
phi_grid = sig.parse("A(x_grid)")

interp["x_grid"] = grid
with torch.no_grad():
    sat = interp(phi_grid).value.cpu().numpy()

plt.figure(figsize=(4, 4))
plt.contourf(gx.numpy(), gy.numpy(), sat.reshape(100, 100), levels=20, cmap="RdYlGn")
plt.colorbar(label="A(x)")
plt.scatter(*pts_te[lbl_te].T,  c="blue",  s=15, label="pos test")
plt.scatter(*pts_te[~lbl_te].T, c="red",   s=15, label="neg test")
plt.legend(fontsize=8); plt.title("Learned A(x)"); plt.tight_layout(); plt.show()


## 5 — Derived predicates (or terms) with `sig.define`

`Signature.define(name, arg_names, formula_body)` registers a rewriting rule: whenever
`sig.parse(...)` encounters `name(t1, …, tn)`, it substitutes `formula_body` with each
`arg_names[i]` replaced by `ti`, entirely at parse time — before symbol resolution or evaluation.

`name` is then usable in further `sig.parse(...)` formulas exactly like any other predicate — no
grounding is needed for it, since the definition expands into its already-grounded components.

```python
sig.define("C", ["z"], "A(z) & B(z)")   # C(z) expands to A(z) & B(z) at parse time
phi = sig.parse("C(z)")                 # no separate grounding for C required
```

### Formula caching

Within a single `interp(phi)` call every sub-formula node caches its result keyed by
`(repr(node), assignment_fingerprint)`. Two *structurally identical* sub-expressions
(same predicate, same argument names) are therefore evaluated **once** even if they
appear multiple times in the formula tree.

The cache is created at the start of `interp(phi)` and discarded at the end — it does
not persist across training steps. The cell below demonstrates this with a predicate that
counts how many times it is actually invoked.


### Example: `sig.define` in practice

The cell below builds a small signature, defines `C(z)` as the rewriting rule `A(z) & B(z)`, and confirms that evaluating `C(z)` and `A(z) & B(z)` directly produce identical truth values.

In [ ]:
from sltn import Signature, Interpretation
import torch

sig2 = Signature("derived_demo")
sig2.sort("Point")
sig2.variable("z", "Point")
sig2.predicate("A", ["Point"])
sig2.predicate("B", ["Point"])
sig2.define("C", ["z"], "A(z) & B(z)")   # declared here; interpretation is set below


interp2 = Interpretation(sig2)
interp2["Point"] = sltn.Type("Point", shape=(2,))
interp2["A"] = lambda x: (x[:, [0]] > 0.5).float()   # A: first coord > 0.5
interp2["B"] = lambda x: (x[:, [1]] < 0.5).float()   # B: second coord < 0.5
# Bind C's interpretation as the formula A(z) & B(z) — no nn.Module needed

phi_C     = sig2.parse("C(z)")
phi_AandB = sig2.parse("A(z) & B(z)")

test_pts = torch.rand(6, 2)
interp2["z"] = test_pts

with torch.no_grad():
    sat_C     = interp2(phi_C).value
    sat_AandB = interp2(phi_AandB).value

print("         z        C(z)   A(z)&B(z)")
for p, c, ab in zip(test_pts, sat_C.squeeze(), sat_AandB.squeeze()):
    print(f"  [{p[0]:.2f}, {p[1]:.2f}]   {c:.2f}    {ab:.2f}")
# C(z) and A(z) & B(z) are identical: the derived grounding correctly delegates.

In [ ]:
call_count = {"A": 0}

class CountedA(torch.nn.Module):
    def forward(self, x):
        call_count["A"] += 1
        return (x[:, [0]] > 0.5).float()

interp2["A"] = CountedA()

formula = "A(z) & A(z) & A(z)"
phi_double = sig2.parse(formula)
interp2["z"] = torch.rand(8, 2)

interp2.use_cache = True

call_count["A"] = 0
with torch.no_grad():
    _ = interp2(phi_double)

print(f"Cache enabled: A evaluated {call_count['A']} time(s) for formula '{formula}'")

interp2.use_cache = False

call_count["A"] = 0
with torch.no_grad():
    _ = interp2(phi_double)

print(f"Cache disabled: A evaluated {call_count['A']} time(s) for formula '{formula}'")

---

**Next:** [Tutorial 5 — Structural LTN](5-StructuralLTN.ipynb) extends everything covered so far with structural axes for reasoning over time, sequences, and graphs.